In [ ]:
from config import settings
from transformers import AutoModelForTokenClassification, AutoTokenizer
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import json
# Load from huggingface

DATASET_REPO_ID = settings.HUGGINGFACE_REPO_ID_BIOBERT
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"

dataset = load_dataset(DATASET_REPO_ID)
# NOTE that the actual labels that will be used for training are under the column: "token_label_ids"
dataset = dataset.rename_column("token_label_ids", "labels")

# Download and load id2label.json from the hub
id2label_path = hf_hub_download(
    repo_id=DATASET_REPO_ID,
    filename="id2label.json",
    repo_type="dataset"
)
# Download and load label2id.json from the hub
label2id_path = hf_hub_download(
    repo_id=DATASET_REPO_ID,
    filename="label2id.json",
    repo_type="dataset"
)
with open(id2label_path, "r") as f:
    id2label = json.load(f)
with open(label2id_path, "r") as f:
    label2id = json.load(f)

# Number of labels 
num_labels = len(id2label)
print("Number of labels:" , num_labels)

# Initialize model, tokenizer
model = AutoModelForTokenClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label, # this would cause the compute_metrics to fail, HF will convert keys t int automatically {int(k): v for k, v in id2label.items()}
    label2id=label2id
    )

Generating test split: 100%|██████████| 1786/1786 [00:00<00:00, 709579.14 examples/s]


Number of labels: 3525


Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 1) Check label id space and consistency

In [9]:
# run this before training (or load your dataset)
import numpy as np
from collections import Counter

labels_all = []
for split in dataset.keys():
    for ex in dataset[split]:
        labels_all.extend(ex["labels"])  # you renamed token_label_ids -> labels

labels_all = np.array(labels_all)
unique, counts = np.unique(labels_all, return_counts=True)
dist = dict(zip(unique.tolist(), counts.tolist()))
print("Unique label ids (including -100):", sorted(dist.keys())[:20], "... (total unique)", len(dist))
print("Counts for first 30 unique ids:", {k: dist[k] for k in sorted(dist.keys())[:30]})
print("Counts for the last 30 unique ids:", {k: dist[k] for k in sorted(dist.keys())[-30:]})
print("Max label id (excluding -100):", max([int(x) for x in unique if int(x) != -100]))
print("model.config.num_labels:", model.config.num_labels)
# sanity checks
max_id = max([int(x) for x in unique if int(x) != -100])
if max_id >= model.config.num_labels:
    print("!! ERROR: max label id >= num_labels. You must remap label ids to 0..num_labels-1")
else:
    print("Label id range OK.")


Unique label ids (including -100): [-100, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18] ... (total unique) 3526
Counts for first 30 unique ids: {-100: 35720, 0: 10, 1: 10, 2: 10, 3: 10, 4: 10, 5: 10, 6: 10, 7: 10, 8: 10, 9: 10, 10: 10, 11: 10, 12: 10, 13: 10, 14: 10, 15: 10, 16: 10, 17: 10, 18: 10, 19: 10, 20: 10, 21: 10, 22: 10, 23: 10, 24: 10, 25: 10, 26: 10, 27: 10, 28: 10}
Counts for the last 30 unique ids: {3495: 20, 3496: 10, 3497: 10, 3498: 70, 3499: 70, 3500: 20, 3501: 20, 3502: 40, 3503: 40, 3504: 30, 3505: 30, 3506: 10, 3507: 10, 3508: 30, 3509: 30, 3510: 40, 3511: 40, 3512: 50, 3513: 50, 3514: 20, 3515: 20, 3516: 30, 3517: 30, 3518: 20, 3519: 20, 3520: 30, 3521: 30, 3522: 10, 3523: 10, 3524: 69654}
Max label id (excluding -100): 3524
model.config.num_labels: 3525
Label id range OK.


**PASS**

# 2) Check label2id / id2label types and mapping

In [5]:
print("id2label sample (type check):", list(id2label.items())[:5])
print("label2id sample (type check):", list(label2id.items())[:5])
# types
print("id2label key types:", set(type(k) for k in id2label.keys()))
print("id2label value types:", set(type(v) for v in id2label.values()))
print("label2id key types:", set(type(k) for k in label2id.keys()))
print("label2id value types:", set(type(v) for v in label2id.values()))

# If id2label keys are strings, convert them:
if any(isinstance(k, str) for k in id2label.keys()):
    print("Converting id2label keys to int...")
    id2label = {int(k): v for k, v in id2label.items()}

# If label2id values are strings, convert them:
if any(isinstance(v, str) for v in label2id.values()):
    print("Converting label2id values to int...")
    label2id = {k: int(v) for k, v in label2id.items()}

# re-check
print("After conversion, model.config.num_labels would be:", len(id2label))


id2label sample (type check): [('0', 'B-SYMPTOM_s0001_NEG'), ('1', 'B-SYMPTOM_s0001_POS'), ('2', 'B-SYMPTOM_s0002_NEG'), ('3', 'B-SYMPTOM_s0002_POS'), ('4', 'B-SYMPTOM_s0003_NEG')]
label2id sample (type check): [('B-SYMPTOM_s0001_NEG', 0), ('B-SYMPTOM_s0001_POS', 1), ('B-SYMPTOM_s0002_NEG', 2), ('B-SYMPTOM_s0002_POS', 3), ('B-SYMPTOM_s0003_NEG', 4)]
id2label key types: {<class 'str'>}
id2label value types: {<class 'str'>}
label2id key types: {<class 'str'>}
label2id value types: {<class 'int'>}
Converting id2label keys to int...
After conversion, model.config.num_labels would be: 3525


# 3) Verify labels survive the DataCollator and batching

In [7]:
from transformers import DataCollatorForTokenClassification
tokenizer_local = tokenizer  # your tokenizer
dc = DataCollatorForTokenClassification(tokenizer_local)

# take a small batch from your dataset (train or validation)
batch_examples = [dataset["train"][i] for i in range(8)]
# If your dataset rows are dicts with numpy arrays, convert them to lists first
# collator expects list of dicts with 'input_ids','attention_mask','labels'
batch_collated = dc(batch_examples)
print("Collated keys:", batch_collated.keys())
print("labels shape:", batch_collated["labels"].shape)
print("labels sample (first two rows):")
print(batch_collated["labels"][:2])
# Count non -100 tokens in the batch
import torch
lab = batch_collated["labels"]
print("Non-ignored tokens per example:", (lab != -100).sum(dim=1).tolist())
print("Unique label ids in batch:", torch.unique(lab[lab != -100]).tolist())


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`text` in this case) have excessive nesting (inputs type `list` where type `int` is expected).